<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 30px;
            border-radius: 12px;
            margin: 25px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h1 style="margin: 0 0 15px 0; color: #ffffff; font-size: 2em; font-weight: 700;">
        ✅ SOLUTIONS - Part 3: Multi-Agent Orchestration
    </h1>
    <h3 style="margin: 0 0 20px 0; color: #ffffff; opacity: 0.95; font-weight: 400; font-size: 1.15em;">
        Complete Reference Implementation
    </h3>
    <div style="background: rgba(255,255,255,0.08);
                backdrop-filter: blur(8px);
                border: 1px solid rgba(255,255,255,0.15);
                padding: 20px;
                border-radius: 8px;
                margin-top: 20px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.15);">
        <p style="margin: 0 0 12px 0; font-size: 1.1em; color: #ffffff;">
            <strong style="color: #ffffff;">🚀 Quick Start:</strong> To see the complete implementation:
        </p>
        <ol style="margin: 10px 0; padding-left: 25px; line-height: 1.8; font-size: 1.05em; color: #ffffff;">
            <li><strong>Select Kernel (Python 3.13.3) → Menu → Run All</strong> (or press Shift+Enter through each cell)</li>
            <li>Watch the notebook execute all setup and solution code</li>
            <li>Review the complete Pricing and Recommendation Agent solutions</li>
            <li>Test the full multi-agent system with the Orchestrator</li>
        </ol>
        <p style="margin: 15px 0 0 0; font-size: 1.02em; color: #ffffff;">
            ⏱️ <strong style="color: #ffffff;">Total Runtime:</strong> ~2-3 minutes
        </p>
    </div>
</div>

---

## ⚙️ Step 0: Select Python Kernel

<div style="background: linear-gradient(135deg, rgba(245,158,11,0.18) 0%, rgba(217,119,6,0.18) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(245,158,11,0.35);
            border-left: 6px solid #f59e0b;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            box-shadow: 0 4px 16px rgba(245,158,11,0.25);">
    <h4 style="margin: 0 0 15px 0; color: #fbbf24; font-size: 1.2em; font-weight: 600;">
        ⚠️ IMPORTANT FIRST STEP
    </h4>
    <p style="margin: 0; color: #e9ecef; line-height: 1.6; font-size: 1.02em;">
        Before running any code, ensure you're using <strong style="color: #fbbf24;">Python 3.13.3</strong> kernel.
    </p>
</div>

In [ ]:
# Verify Python Environment
import sys

python_version = sys.version.split()[0]
required_version = "3.13"

print(f"🐍 Python Version: {python_version}")
print("─" * 50)

if python_version.startswith(required_version):
    print("✅ Environment Verified")
    print(f"   Running Python {python_version} (Required: {required_version}.x)")
else:
    print(f"⚠️  Environment Mismatch Detected")
    print(f"   Current: Python {python_version}")
    print(f"   Required: Python {required_version}.x")

---

# Setup: Environment & Tools

In [ ]:
# ============================================================================
# Environment Setup: Database Connection & Bedrock Client
# ============================================================================

import os
import json
import psycopg
from psycopg.rows import dict_row
import boto3
from typing import List, Dict
from dotenv import load_dotenv
from strands import Agent, tool
import logging

# Load environment
load_dotenv('/workshop/sample-dat406-build-agentic-ai-powered-search-apg/.env')

# Database configuration
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'postgres')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Bedrock client
bedrock = boto3.client('bedrock-runtime', region_name=os.environ.get('AWS_REGION', 'us-west-2'))

print("✅ Environment configured")
print(f"   Database: {DB_NAME}")
print(f"   Region: {bedrock.meta.region_name}")

In [ ]:
def generate_embedding(text: str) -> List[float]:
    """Generate embeddings using Amazon Titan Text Embeddings V2."""
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    result = json.loads(response['body'].read())
    return result['embedding']

print("✅ Embedding function ready")

## Load All 7 Tools from Part 2

**Tools by Agent:**
- **Inventory Agent:** `get_inventory_health`, `restock_product`, `get_low_stock_products``
- **Pricing Agent:** `get_category_price_analysis`, `get_product_by_category`, `semantic_product_search`
- **Recommendation Agent:** `get_trending_products`, `semantic_product_search`, `get_product_by_category`

In [ ]:
# ============================================================================
# Complete Tool Suite from Part 2
# ============================================================================

@tool
def get_inventory_health() -> str:
    """Retrieves current inventory health statistics."""
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT 
                        COUNT(*) as total_products,
                        COUNT(CASE WHEN quantity = 0 THEN 1 END) as out_of_stock,
                        COUNT(CASE WHEN quantity > 0 AND quantity < 10 THEN 1 END) as low_stock,
                        COUNT(CASE WHEN quantity >= 10 THEN 1 END) as healthy_stock,
                        AVG(quantity) as avg_quantity,
                        SUM(quantity) as total_quantity
                    FROM bedrock_integration.product_catalog
                """)
                stats = dict(cur.fetchone())
                
                total = stats['total_products']
                healthy = stats['healthy_stock']
                health_score = int((healthy / total * 100)) if total > 0 else 0
                
                cur.execute("""
                    SELECT "productId", product_description, stars, reviews, quantity
                    FROM bedrock_integration.product_catalog
                    WHERE quantity < 10 AND stars >= 4.0 AND reviews > 100
                    ORDER BY quantity ASC, reviews DESC LIMIT 10
                """)
                critical_items = [dict(row) for row in cur.fetchall()]
                
                alerts = []
                if stats['out_of_stock'] > 0:
                    alerts.append(f"🚨 {stats['out_of_stock']} products out of stock")
                if stats['low_stock'] > 100:
                    alerts.append(f"⚠️ {stats['low_stock']} products low stock (<10 units)")
                elif stats['low_stock'] > 0:
                    alerts.append(f"⚠️ {stats['low_stock']} products need monitoring")
                if not alerts:
                    alerts.append("✅ Inventory healthy")
                
                result = {
                    "health_score": health_score,
                    "statistics": {
                        "total_products": stats['total_products'],
                        "out_of_stock": stats['out_of_stock'],
                        "low_stock": stats['low_stock'],
                        "healthy_stock": stats['healthy_stock'],
                        "avg_quantity": round(float(stats['avg_quantity']), 2),
                        "total_quantity": int(stats['total_quantity'])
                    },
                    "critical_items": critical_items,
                    "alerts": alerts
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Inventory check error: {str(e)}"})

@tool
def get_low_stock_products(limit: int = 3) -> str:
    """
    Get products with low stock (quantity < 10) prioritized by demand.
    Used by Inventory Agent to identify items needing restocking.
    
    Args:
        limit: Number of results (default: 3)
    
    Returns:
        JSON with low-stock products sorted by urgency (lowest quantity first)
    """
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT 
                        "productId",
                        product_description,
                        price,
                        stars,
                        reviews,
                        category_name,
                        quantity,
                        "imgUrl",
                        "productURL"
                    FROM bedrock_integration.product_catalog
                    WHERE quantity < 10
                      AND stars >= 3.0
                    ORDER BY quantity ASC, reviews DESC, stars DESC
                    LIMIT %s
                """, (limit,))
                
                products = [dict(row) for row in cur.fetchall()]
                
                result = {
                    "status": "success",
                    "count": len(products),
                    "products": [{
                        "id": p["productId"],
                        "name": p["product_description"],
                        "category": p["category_name"],
                        "price": float(p["price"]),
                        "rating": float(p["stars"]),
                        "reviews": p["reviews"],
                        "quantity": p["quantity"],
                        "imgUrl": p.get("imgUrl", ""),
                        "productURL": p.get("productURL", "#")
                    } for p in products]
                }
                
                return json.dumps(result, indent=2)
                
    except Exception as e:
        return json.dumps({"error": f"Low stock check error: {str(e)}"})

@tool
def get_trending_products(limit: int = 10) -> str:
    """Get trending products based on ratings and reviews."""
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT "productId", product_description, category_name, price, 
                           stars, reviews, quantity, "imgUrl", "productURL"
                    FROM bedrock_integration.product_catalog
                    WHERE stars >= 4.5 AND reviews > 50 AND quantity > 0
                    ORDER BY stars DESC, reviews DESC LIMIT %s
                """, (limit,))
                products = cur.fetchall()
                
                result = {
                    "trending_products": [{
                        "id": p["productId"],
                        "name": p["product_description"],
                        "category": p["category_name"],
                        "price": float(p["price"]),
                        "rating": float(p["stars"]),
                        "reviews": p["reviews"],
                        "imgUrl": p.get("imgUrl", ""),
                        "productURL": p.get("productURL", "#")
                    } for p in products],
                    "count": len(products),
                    "criteria": "stars >= 4.5, reviews > 50, in stock"
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Database error: {str(e)}"})


@tool
def get_category_price_analysis(category: str) -> str:
    """Analyze pricing statistics for a product category."""
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT category_name, COUNT(*) as total_products,
                           MIN(price) as min_price, MAX(price) as max_price,
                           AVG(price) as avg_price, AVG(stars) as avg_rating
                    FROM bedrock_integration.product_catalog
                    WHERE category_name ILIKE %s AND quantity > 0
                    GROUP BY category_name
                """, (f"%{category}%",))
                row = cur.fetchone()
                
                if not row:
                    return json.dumps({"error": f"No products found in category: {category}"})
                
                cur.execute("""
                    SELECT "productId", product_description, price, stars, reviews,
                           "imgUrl", "productURL", (price / NULLIF(stars, 0)) as value_score
                    FROM bedrock_integration.product_catalog
                    WHERE category_name ILIKE %s AND quantity > 0 AND stars > 0 AND price > 0
                    ORDER BY value_score ASC LIMIT 5
                """, (f"%{category}%",))
                best_value = cur.fetchall()
                
                result = {
                    "category": row["category_name"],
                    "analysis": {
                        "total_products": row["total_products"],
                        "price_range": {
                            "min": float(row["min_price"]),
                            "max": float(row["max_price"]),
                            "average": round(float(row["avg_price"]), 2)
                        },
                        "average_rating": round(float(row["avg_rating"]), 2)
                    },
                    "best_value_products": [{
                        "id": p["productId"],
                        "name": p["product_description"],
                        "price": float(p["price"]),
                        "rating": float(p["stars"]),
                        "reviews": p["reviews"],
                        "imgUrl": p.get("imgUrl", ""),
                        "productURL": p.get("productURL", "#"),
                        "value_score": round(float(p["value_score"]), 2)
                    } for p in best_value]
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Analysis error: {str(e)}"})


@tool
def restock_product(product_id: str, quantity: int) -> str:
    """Restock a product by updating its quantity."""
    try:
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT "productId", product_description, quantity
                    FROM bedrock_integration.product_catalog WHERE "productId" = %s
                """, (product_id,))
                product = cur.fetchone()
                
                if not product:
                    return json.dumps({"status": "error", "message": f"Product {product_id} not found"})
                
                old_quantity = product['quantity']
                new_quantity = old_quantity + quantity
                
                cur.execute("""
                    UPDATE bedrock_integration.product_catalog
                    SET quantity = quantity + %s WHERE "productId" = %s
                """, (quantity, product_id))
                conn.commit()
                
                result = {
                    "status": "success",
                    "product_id": product_id,
                    "product_name": product['product_description'],
                    "old_quantity": old_quantity,
                    "added_quantity": quantity,
                    "new_quantity": new_quantity,
                    "message": f"✅ Added {quantity} units to {product['product_description']}"
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Restock error: {str(e)}"})


@tool
def semantic_product_search(query: str, max_price: float = None,
                            min_rating: float = 4.0, category: str = None,
                            limit: int = 5) -> str:
    """AI-powered semantic search from Part 1."""
    try:
        query_embedding = generate_embedding(query)
        if not query_embedding:
            return json.dumps({"error": "Failed to generate embedding"})
        
        conditions = ["quantity > 0"]
        params = [str(query_embedding), str(query_embedding)]
        
        if max_price:
            conditions.append("price <= %s")
            params.append(max_price)
        if min_rating:
            conditions.append("stars >= %s")
            params.append(min_rating)
        if category:
            conditions.append("category_name ILIKE %s")
            params.append(f"%{category}%")
        
        where_clause = " AND ".join(conditions)
        params.append(str(query_embedding))
        params.append(limit)
        
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute(f"""
                    SELECT "productId", product_description, price, stars, reviews,
                           category_name, quantity, "imgUrl", "productURL",
                           1 - (embedding <=> %s::vector) as similarity
                    FROM bedrock_integration.product_catalog
                    WHERE {where_clause}
                    ORDER BY embedding <=> %s::vector LIMIT %s
                """, params)
                products = cur.fetchall()
                
                result = {
                    "query": query,
                    "matches": [{
                        "id": p["productId"],
                        "name": p["product_description"],
                        "category": p["category_name"],
                        "price": float(p["price"]),
                        "rating": float(p["stars"]),
                        "reviews": p["reviews"],
                        "imgUrl": p.get("imgUrl", ""),
                        "productURL": p.get("productURL", "#"),
                        "similarity": round(float(p["similarity"]), 3)
                    } for p in products],
                    "count": len(products),
                    "filters": {"max_price": max_price, "min_rating": min_rating, "category": category}
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Search error: {str(e)}"})


@tool
def get_product_by_category(category: str, min_rating: float = 4.0,
                            max_price: float = None, limit: int = 10) -> str:
    """Retrieve products by category with filters."""
    try:
        conditions = ["category_name ILIKE %s", "quantity > 0"]
        params = [f"%{category}%"]
        
        if min_rating:
            conditions.append("stars >= %s")
            params.append(min_rating)
        if max_price:
            conditions.append("price <= %s")
            params.append(max_price)
        
        where_clause = " AND ".join(conditions)
        params.append(limit)
        
        with psycopg.connect(conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute(f"""
                    SELECT "productId", product_description, price, stars, reviews,
                           category_name, quantity, "imgUrl", "productURL"
                    FROM bedrock_integration.product_catalog
                    WHERE {where_clause}
                    ORDER BY stars DESC, reviews DESC LIMIT %s
                """, params)
                products = cur.fetchall()
                
                result = {
                    "category": category,
                    "products": [{
                        "id": p["productId"],
                        "name": p["product_description"],
                        "category": p["category_name"],
                        "price": float(p["price"]),
                        "rating": float(p["stars"]),
                        "reviews": p["reviews"],
                        "quantity": p["quantity"],
                        "imgUrl": p.get("imgUrl", ""),
                        "productURL": p.get("productURL", "#")
                    } for p in products],
                    "count": len(products),
                    "filters": {"min_rating": min_rating, "max_price": max_price}
                }
                return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Category search error: {str(e)}"})


print("✅ All 7 tools from Part 2 loaded")
print("   • Inventory: get_inventory_health, restock_product, get_low_stock_products")
print("   • Pricing: get_category_price_analysis, get_product_by_category, semantic_product_search")
print("   • Recommendation: get_trending_products, semantic_product_search, get_product_by_category")

---

# Specialist Agents - Complete Solutions

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            border-left: 6px solid #28a745;
            padding: 25px;
            border-radius: 12px;
            margin: 20px 0;
            box-shadow: 0 8px 20px rgba(40,167,69,0.25);">
    <h2 style="margin: 0 0 15px 0; color: #ffffff; font-weight: 600;">
        ✅ All Three Specialist Agents
    </h2>
    <p style="margin: 0; color: #ffffff; font-size: 1.1em; line-height: 1.6; opacity: 0.95;">
        Complete implementations for Inventory, Pricing, and Recommendation agents.
    </p>
</div>

## Inventory Agent

In [ ]:
# ============================================================================
# Inventory Agent - Complete Implementation
# ============================================================================

inventory_agent = Agent(
    name="Inventory Specialist",
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="""You are Blaize Bazaar's Inventory Specialist.

Key Responsibilities:
- Monitor inventory health and stock levels
- Execute restocking operations for low-stock items
- Analyze product availability by category
- Flag critical stock alerts

Communication Guidelines:
- Provide clear, data-driven responses
- Highlight critical stock alerts
- Recommend restocking actions when needed
- Focus on actionable inventory insights
""",
    tools=[get_inventory_health, restock_product, get_low_stock_products]
)

print("✅ Inventory Agent initialized")
print("   Model: claude-sonnet-4-20250514")
print("   Tools: get_inventory_health, restock_product, get_low_stock_products")

## Pricing Agent - SOLUTION

In [ ]:
# ============================================================================
# SOLUTION: Pricing Agent - Complete Implementation
# ============================================================================

pricing_agent = Agent(
    name="Pricing Specialist",
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="""You are Blaize Bazaar's Pricing Specialist.

Key Responsibilities:
- Analyze pricing statistics across categories
- Identify best value products for customers
- Provide price range insights and budget recommendations
- Use semantic search to find deals matching customer needs

Communication Guidelines:
- Provide clear, budget-conscious recommendations
- Highlight best value products (price per star)
- Explain price ranges in customer-friendly terms
- Help customers find quality products within their budget
""",
    tools=[get_category_price_analysis, get_product_by_category, semantic_product_search]
)

print("✅ Pricing Agent initialized")
print("   Model: claude-sonnet-4-20250514")
print("   Tools: get_category_price_analysis, get_product_by_category, semantic_product_search")

## Recommendation Agent - SOLUTION

In [ ]:
# ============================================================================
# SOLUTION: Recommendation Agent - Complete Implementation
# ============================================================================

recommendation_agent = Agent(
    name="Recommendation Specialist",
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="""You are Blaize Bazaar's Recommendation Specialist.

Key Responsibilities:
- Understand customer needs through natural language
- Use semantic search to match products by meaning, not just keywords
- Suggest trending and popular products
- Provide personalized recommendations based on customer context

Communication Guidelines:
- Ask clarifying questions to understand customer needs
- Use semantic search for intent-based matching
- Explain why recommended products are good fits
- Provide diverse options across price ranges when possible
""",
    tools=[get_trending_products, semantic_product_search, get_product_by_category]
)

print("✅ Recommendation Agent initialized")
print("   Model: claude-sonnet-4-20250514")
print("   Tools: get_trending_products, semantic_product_search, get_product_by_category")

---

# Orchestrator Agent

**The "brain" that coordinates all specialist agents.**

In [ ]:
# ============================================================================
# Orchestrator Agent - Multi-Agent Coordination
# ============================================================================

logging.getLogger('strands').setLevel(logging.ERROR)

orchestrator = Agent(
    name="Customer Service Orchestrator",
    system_prompt="""You are Blaize Bazaar's Customer Service Orchestrator. 
You coordinate specialist agents to deliver intelligent customer support.

Core Responsibilities:
- Analyze queries to determine appropriate specialist(s)
- Route to Inventory, Pricing, or Recommendation agents
- Coordinate multiple specialists for complex queries
- Synthesize insights into clear, conversational responses

Routing Strategy:
- Inventory/stock questions → Inventory Specialist
- Pricing/budget/value → Pricing Specialist  
- Product recommendations → Recommendation Specialist
- Complex queries → Coordinate multiple specialists

Communication Guidelines:
- Natural, customer-friendly responses
- Synthesize specialist outputs seamlessly
- Ask clarifying questions when needed
- Respond as one expert, not multiple systems
""",
    # "Agents as Tools" pattern - specialist agents become callable tools
    tools=[inventory_agent, pricing_agent, recommendation_agent]
)

print("✅ Orchestrator Agent initialized")
print("   Model: claude-sonnet-4-20250514")
print("   Pattern: Agents as Tools")
print("   Coordinates: Inventory, Pricing, Recommendation Agents")

---

# Test the Complete System

**Try different queries to see how the Orchestrator routes to specialists:**
- "What laptops do you have under $1000?"
- "Check inventory levels for Electronics"
- "What's the price range for headphones?"
- "I need wireless headphones for running"
- "Show me trending products"

In [ ]:
# Test the complete multi-agent system

response = orchestrator("What laptops do you have under $1000?")
print("\n🤖 Orchestrator Response:")
print(response)

In [ ]:
# Test inventory routing

response = orchestrator("Check inventory levels for Electronics")
print("\n🤖 Orchestrator Response:")
print(response)

In [ ]:
# Test semantic recommendation

response = orchestrator("I need wireless headphones for running")
print("\n🤖 Orchestrator Response:")
print(response)

---

<div style="background: linear-gradient(135deg, rgba(40,167,69,0.18) 0%, rgba(34,139,34,0.12) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(40,167,69,0.3);
            padding: 20px;
            border-radius: 10px;
            margin: 30px 0;
            text-align: center;
            box-shadow: 0 6px 16px rgba(40,167,69,0.25);">
    <h3 style="margin: 0 0 10px 0; color: #ffffff; font-size: 1.4em; font-weight: 600;">
        🎉 Solutions Complete!
    </h3>
    <p style="margin: 0; color: #ffffff; font-size: 1.05em; line-height: 1.6; opacity: 0.95;">
        You've seen complete implementations for all specialist agents and orchestration.<br>
        <strong style="color: #ffffff;">Ready to take it to the next level?</strong> Explore further with the Blaize Bazaar demo application!
    </p>
</div>

---

<div style="text-align: center; color: #888; font-size: 0.9em; margin-top: 30px;">
Built for AWS re:Invent 2025 | DAT406 Workshop
</div>